# 03 · De HDFS a MinIO con SQL

Ejecuta primero 01 y 02. Usaremos el cliente S3 de Python para explorar MinIO y los motores SQL para transformar los datos. No necesitas abrir una terminal.

In [ ]:
from pathlib import Path
from uuid import uuid4
from clientes import ALUMNO, DB, hive_sql, trino_sql, minio
s3 = minio()
clave = f"aula/{ALUMNO}/ventas_csv/ventas.csv"
s3.upload_file(str(Path("datos") / ALUMNO / "ventas.csv"), "raw-data", clave)
print(s3.get_object(Bucket="raw-data", Key=clave)["Body"].read().decode())

## Registrar el CSV de MinIO y leerlo con Hive y Trino
`raw-data` es un bucket y el resto de la ruta es el prefijo. `s3a://` señala almacenamiento S3; no es una ruta HDFS.

In [ ]:
hive_sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS {DB}.ventas_minio (
 id INT, ciudad STRING, importe DOUBLE
) ROW FORMAT DELIMITED FIELDS TERMINATED BY ","
STORED AS TEXTFILE
LOCATION 's3a://raw-data/aula/{ALUMNO}/ventas_csv'""")
for ejecutar, tabla in [(hive_sql, f"{DB}.ventas_minio"), (trino_sql, f"hive.{DB}.ventas_minio")]:
    resultado = ejecutar(f"SELECT COUNT(*) AS filas FROM {tabla}")
    display(resultado)
    assert int(resultado.iloc[0, 0]) == 3

## Transformar el CSV de HDFS y escribir Parquet en MinIO
Cada ejecución crea una tabla nueva con un sufijo. Así no borra ni duplica filas en resultados anteriores. Todas quedan disponibles para explorarlas.

In [ ]:
esquema = f"{DB}_s3"
trino_sql(f"CREATE SCHEMA IF NOT EXISTS hive.{esquema} WITH (location = 's3a://warehouse/aula/{ALUMNO}/')")
tabla = "resumen_" + uuid4().hex[:8]
trino_sql(f"""CREATE TABLE hive.{esquema}.{tabla}
WITH (format = 'PARQUET') AS
SELECT ciudad, SUM(importe) AS total
FROM hive.{DB}.ventas_csv GROUP BY ciudad""")
resultado = trino_sql(f"SELECT * FROM hive.{esquema}.{tabla} ORDER BY ciudad")
display(resultado)
assert resultado.iloc[:, 1].tolist() == [150.0, 200.0]
print("Tabla creada:", f"{esquema}.{tabla}")

## Leer el Parquet con Hive y observar los objetos
Esto demuestra que el resultado no pertenece exclusivamente a Trino: Hive puede descubrir la tabla y leer sus archivos.

In [ ]:
resultado_hive = hive_sql(f"SELECT * FROM {esquema}.{tabla} ORDER BY ciudad")
display(resultado_hive)
assert resultado_hive.values.tolist() == resultado.values.tolist()
objetos = s3.list_objects_v2(Bucket="warehouse", Prefix=f"aula/{ALUMNO}/").get("Contents", [])
for objeto in objetos:
    print(objeto["Key"], objeto["Size"], "bytes")
assert objetos

También puedes abrir la consola MinIO en http://localhost:9001 (`minioadmin` / `minioadmin`). Sirve para explorar buckets y objetos, no para escribir SQL.

**Ejercicio:** crea otra tabla Parquet que conserve solo las ventas mayores de 75 y comprueba sus filas desde ambos motores.